# Variational Autoencoder (VAE)

_[based on our assignment from YSDA [DeepCVG course](https://github.com/yandexdataschool/deep_vision_and_graphics/tree/fall23); materials by Victor Yurchenko]_

**TL;DR this task** will walk involve you implementing your variational autoencoder with normal latent variables - just the basic variety - to model celebrity faces from CelebA dataset.

![img](https://miro.medium.com/v2/resize:fit:1400/0*39kjKwCtVjhrvsQ7.png)


Source links (in addition to our materials):
* Original paper http://arxiv.org/abs/1312.6114
* Helpful videos explaining the topic
   * https://www.youtube.com/watch?v=P78QYjWh5sM     
   * http://videolectures.net/deeplearning2015_courville_autoencoder_extension/?q=aaron%20courville

### Download data

In [ ]:
#changed
import os

if not os.path.exists('yfile.py'):
  !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_DL/refs/heads/fall24/week09_generative/yfile.py -O yfile.py
from yfile import download_from_yadisk

TARGET_DIR = '.'
FILENAME = "celeba.zip"

if not os.path.exists(os.path.join(TARGET_DIR, FILENAME)):
  print("Downloading...") # this will download 1.4 gb file, downloading will take some time (8-10 minutes)
  download_from_yadisk(short_url='https://disk.yandex.ru/d/QErNx-MoC-2Y6Q',
                        filename=FILENAME, target_dir=TARGET_DIR)
  # alternative manual download:  https://drive.google.com/file/d/1lwn2JMwWNGloGvyzKUOcewT8JqP1ICoW
else:
  print("Found previously downloaded data!")

filesize = os.path.getsize(os.path.join(TARGET_DIR, FILENAME))
assert filesize > 2 ** 30, f"{filesize / 2 ** 30:.2f} GiB is too small, something wrong with downloading - please clean up and re-download"

# checked extraction separately from the downloaded archive
required_files = [
    'celeba/celeba/img_align_celeba/100019.jpg',
    'celeba/celeba/img_align_celeba/000084.jpg',
    'celeba/celeba/img_align_celeba/148483.jpg',
    'celeba/celeba/list_eval_partition.txt',
    'celeba/celeba/list_attr_celeba.txt',
    'celeba/celeba/identity_CelebA.txt',
    'celeba/celeba/list_bbox_celeba.txt',
    'celeba/celeba/list_landmarks_align_celeba.txt',
]
if not all(os.path.exists(path) for path in required_files):
  print("Extracting...")
  !unzip -q -n celeba.zip  # extract images; you don't need to run it every time
  print("Done!")
assert all(os.path.exists(path) for path in required_files), 'Check the extracted CelebA directory.'

import matplotlib.pyplot as plt
plt.subplot(131); plt.imshow(plt.imread("celeba/celeba/img_align_celeba/100019.jpg"))
plt.subplot(132); plt.imshow(plt.imread("celeba/celeba/img_align_celeba/000084.jpg"))
plt.subplot(133); plt.imshow(plt.imread("celeba/celeba/img_align_celeba/148483.jpg"))


## Prepare the data

In [ ]:
#changed
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision
from torchvision.utils import make_grid

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
from pathlib import Path
import time
from tqdm.auto import tqdm

# fixed the seed and kept checkpoints in one directory
np.random.seed(1337)
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1337)
checkpoint_dir = Path('checkpoints_vae')
checkpoint_dir.mkdir(exist_ok=True)
print('Device:', device)


In [ ]:
# Note that for simplicity we reduce the number of channels to 1
# But if you want to be hardcore feel free to comment-out .Grayscale below :)


class CropCelebA64:  # custom preprocessing for CelebA
    def __call__(self, pic):
        new_pic = pic.crop((15, 40, 178 - 15, 218 - 30))
        return new_pic
    def __repr__(self):
        return self.__class__.__name__ + '()'


train_dataset = torchvision.datasets.CelebA(
    root='celeba',
    split='train',
    transform=torchvision.transforms.Compose([
        CropCelebA64(),
        torchvision.transforms.Resize(64),
        torchvision.transforms.RandomHorizontalFlip(),
        torchvision.transforms.ToTensor(),

        torchvision.transforms.Grayscale()
    ]),
    #download=True,   # in case of problems with downloading from gdrive above
)

validation_dataset = torchvision.datasets.CelebA(
    root='celeba',
    split='valid',
    transform=torchvision.transforms.Compose([
        CropCelebA64(),
        torchvision.transforms.Resize(64),
        torchvision.transforms.ToTensor(),

        torchvision.transforms.Grayscale()
    ]),
)

In [ ]:
#changed
# Hold out the first 10k images for evaluation
num_images = 10000
# used deterministic preprocessing for the held-out images
evaluation_dataset = torchvision.datasets.CelebA(
    root='celeba', split='train', transform=validation_dataset.transform
)
real_images_tensor = torch.stack([evaluation_dataset[i][0] for i in range(num_images)], dim=0)
real_images = real_images_tensor.numpy().transpose([0, 2, 3, 1])
np.savez("real.npz", Pictures=real_images.reshape(num_images, 64 * 64 * 1))

# removed the held-out images from training
train_dataset = Subset(train_dataset, range(num_images, len(train_dataset)))
print('Training images:', len(train_dataset), '| validation images:', len(validation_dataset))
del real_images_tensor, real_images, evaluation_dataset


In [ ]:
BATCH_SIZE = 64  # feel free to alter after initial tuning

# you may find useful parameters `num_workers` and `pin_memory`
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE)
print("Image batch shape:", next(iter(train_loader))[0].shape)


# Autoencoder

Why to use all this complicated formulaes and regularizations, what is the need for variational inference? To analyze the difference, let's first train just an autoencoder on the data:

### implementation choices

I followed the explicit instruction to use two fully connected layers per encoder/decoder, with a 512-unit ReLU hidden layer. The AE template also mentioned a convnet and Gaussian sampling, which conflicted with this section's ordinary-autoencoder baseline. I kept the AE deterministic and used reparameterization only in the VAE. The original comments were retained for reference.

I kept grayscale 64x64 inputs, `BATCH_SIZE = 64`, `dimZ = 100` and Adam (default learning rate 0.001). Since the starter did not specify an epoch count, I set 10 epochs for each model. I excluded the first 10,000 training-split images from optimization to follow the hold-out instruction and kept the official validation split for model selection. The reference images saved to `real.npz` use deterministic transforms.

For the VAE, I kept the same hidden width, learned both pixel means and log standard deviations, and used one latent sample per input for the ELBO. I bounded predicted log standard deviations to [-6, 2] and clipped VAE gradient norms at 5 for numerical stability. Reconstruction previews use the posterior mean; random generation uses both latent and pixel noise.

**Run status:** the models have not been trained in this completed notebook yet. Run on GPU and inspect the losses and images before drawing conclusions. Best-validation weights and the latest training state are saved under `checkpoints_vae` after each epoch. In Colab, download these files or copy them to persistent storage before the runtime ends; the local runtime directory is not a backup.

In [ ]:
#changed
dimZ = 100  # Considering face reconstruction task, which size of representation seems reasonable?

# Define the decoder and encoder as networks with one hidden fc-layer
# (that means you will have 2 fc layers in each net)
# Use ReLU for hidden layers' activations
# GlorotUniform initialization for W
# Zero initialization for biases
# It's also convenient to put sigmoid activation on output layer to get nice normalized pics


class Autoencoder(nn.Module):

    def __init__(self):
        super(Autoencoder, self).__init__()

        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 64, 512),
            nn.ReLU(),
            nn.Linear(512, dimZ),
        )
        self.decoder = nn.Sequential(
            nn.Linear(dimZ, 512),
            nn.ReLU(),
            nn.Linear(512, 64 * 64),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 64, 64)),
        )
        for layer in self.modules():
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        # note: use reparameterization trick, z = mu + randn_like(mu) * e ^ log_sigma
        # kept this AE deterministic; used the trick only in VAE below
        latent_code = self.encoder(x)
        reconstruction = self.decoder(latent_code)
        return reconstruction, latent_code


In [ ]:
#changed
# Create MSE loss function
criterion = torch.nn.MSELoss()

autoencoder = Autoencoder().to(device)

# Use Adam optimizer
optimizer = optim.Adam(autoencoder.parameters())

# checked the AE output shapes before training
with torch.no_grad():
    test_images = torch.zeros(2, 1, 64, 64, device=device)
    test_reconstruction, test_code = autoencoder(test_images)
assert test_reconstruction.shape == test_images.shape
assert test_code.shape == (2, dimZ)


In [ ]:
#changed
# Train your autoencoder
# Visualize progress in reconstruction and loss decay
num_epochs = 10
ae_history = {'train_mse': [], 'val_mse': []}
best_ae_loss = float('inf')
preview_images = next(iter(val_loader))[0][:8].to(device)

for epoch in range(num_epochs):
    start_time = time.time()
    autoencoder.train()
    train_loss, train_count = 0.0, 0
    for images, _ in tqdm(train_loader, desc=f'AE {epoch + 1}/{num_epochs}', leave=False):
        images = images.to(device)
        optimizer.zero_grad()
        reconstruction, latent_code = autoencoder(images)
        loss = criterion(reconstruction, images)
        assert torch.isfinite(loss).item(), 'AE loss became non-finite.'
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        train_count += images.size(0)

    autoencoder.eval()
    val_loss, val_count = 0.0, 0
    with torch.no_grad():
        for images, _ in val_loader:
            images = images.to(device)
            reconstruction, _ = autoencoder(images)
            val_loss += criterion(reconstruction, images).item() * images.size(0)
            val_count += images.size(0)
        preview_reconstruction, _ = autoencoder(preview_images)

    train_loss /= train_count
    val_loss /= val_count
    assert np.isfinite(val_loss), 'AE validation loss became non-finite.'
    ae_history['train_mse'].append(train_loss)
    ae_history['val_mse'].append(val_loss)
    if val_loss < best_ae_loss:
        best_ae_loss = val_loss
        torch.save(autoencoder.state_dict(), checkpoint_dir / 'ae_best.pt')
    torch.save({
        'epoch': epoch + 1, 'model_state_dict': autoencoder.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(), 'history': ae_history,
    }, checkpoint_dir / 'ae_last.pt')

    print(f'AE epoch {epoch + 1}: train MSE = {train_loss:.6f}, '
          f'val MSE = {val_loss:.6f}, time = {time.time() - start_time:.1f}s')
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(range(1, epoch + 2), ae_history['train_mse'], label='train')
    axes[0].plot(range(1, epoch + 2), ae_history['val_mse'], label='validation')
    axes[0].set(xlabel='epoch', ylabel='MSE per pixel', title='AE loss')
    axes[0].legend()
    grid = make_grid(torch.cat([preview_images, preview_reconstruction]).cpu(), nrow=8)
    axes[1].imshow(grid.permute(1, 2, 0))
    axes[1].set_title('AE: originals (top), reconstructions (bottom)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

# restored the checkpoint with the lowest validation MSE
autoencoder.load_state_dict(torch.load(checkpoint_dir / 'ae_best.pt', map_location=device, weights_only=True))
autoencoder.eval()
print(f'Best AE validation MSE: {best_ae_loss:.6f}')


In [ ]:
#changed
# Examine the reconstructions
from itertools import islice

images, _ = list(islice(val_loader, 1))[0]
autoencoder.eval()
with torch.no_grad():
    reconstruction, latent_code = autoencoder(images.to(device))

reconstruction = reconstruction[:16].cpu()
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(make_grid(images[:16], nrow=4).permute(1, 2, 0))
axes[0].set_title('Originals')
axes[1].imshow(make_grid(reconstruction, nrow=4).permute(1, 2, 0))
axes[1].set_title('AE reconstructions')
for ax in axes:
    ax.axis('off')
plt.show()
plt.close(fig)


Reconstruction is not bad, right?  Okay, it very well might be bad, but it should at least be a blurry image.

# Sampling

Let's now sample several latent vectors and perform inference from $z$, reconstruct an image given some random $z$ representations.

In [ ]:
#changed
z = torch.randn(25, dimZ) * 0.5
autoencoder.eval()
with torch.no_grad():
    reconstruction = autoencoder.decoder(z.to(device)).clamp(0, 1)

plt.figure(figsize=(6, 6))
plt.imshow(make_grid(reconstruction.cpu(), nrow=5).permute(1, 2, 0))
plt.title('AE: random latent vectors')
plt.axis('off')
plt.show()


So, if we sample $z$ from normal, whould we eventually generate all possible faces? What do you think?

No. The ordinary AE was trained to reconstruct images, not to make its latent codes follow a standard normal distribution. Random normal vectors can fall in regions that the decoder did not see during training, so their outputs need not be plausible faces. Even a VAE does not guarantee coverage of every possible face.

# Variational Autoencoder

Bayesian approach in deep learning considers everything in terms of distributions. Now our encoder generates not just a vector $z$ but posterior ditribution $q(z|x)$. In our case distribution $q$ is Gaussian distibution $N(\mu, \sigma)$ with parameters $\mu$, $\sigma$. Technically, the first difference is that you need to split bottleneck layer in two. One dense layer will generate vector $\mu$, and another will generate vector $\sigma$. Reparametrization trick should be implemented using the **gaussian_sampler** layer, that generates random vetor $\epsilon$ and returns $\mu+\sigma\epsilon \sim N(\mu, \sigma)$ .

Since our decoder is also a function that generates distribution, we need to do the same splitting for output layer. When testing the model we will look only on mean values, so one of the output will be actual autoencoder output.

In this homework we only ask for implementation of the simplest version of VAE - one $z$ sample per input. You can consider to sample several outputs from one input and average them.

In [ ]:
#changed
# to compare with conventional AE, keep these hyperparameters
# or change them for the values that you used before
dimZ = 100

# define the network
# you can start from https://github.com/pytorch/examples/blob/master/vae/main.py
# or Theano-based examples here https://github.com/Lasagne/Recipes/blob/master/examples/variational_autoencoder/variational_autoencoder.py
# and here https://github.com/y0ast/Variational-Autoencoder/blob/master/VAE.py
# but remember that this is not your ground truth since the data is not MNIST


class VAE(nn.Module):

    def __init__(self):
        super(VAE, self).__init__()

        self.encoder = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 64, 512), nn.ReLU()
        )
        self.encoder_mu = nn.Linear(512, dimZ)
        self.encoder_logsigma = nn.Linear(512, dimZ)
        self.decoder = nn.Sequential(nn.Linear(dimZ, 512), nn.ReLU())
        self.decoder_mu = nn.Linear(512, 64 * 64)
        self.decoder_logsigma = nn.Linear(512, 64 * 64)

        for layer in self.modules():
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def encode(self, x):
        hidden = self.encoder(x)
        mu = self.encoder_mu(hidden)
        # bounded log standard deviations to avoid extreme variances
        logsigma = self.encoder_logsigma(hidden).clamp(-6, 2)
        return mu, logsigma

    def decode(self, z):
        hidden = self.decoder(z)
        mu = torch.sigmoid(self.decoder_mu(hidden)).reshape(-1, 1, 64, 64)
        logsigma = self.decoder_logsigma(hidden).clamp(-6, 2).reshape(-1, 1, 64, 64)
        return mu, logsigma

    def gaussian_sampler(self, mu, logsigma):
        noise = torch.randn_like(mu)
        return noise * logsigma.exp() + mu

    def forward(self, x):
        latent_mu, latent_logsigma = self.encode(x)
        z = self.gaussian_sampler(latent_mu, latent_logsigma)
        reconstruction_mu, reconstruction_logsigma = self.decode(z)
        # return reconstruction_mu, reconstruction_logsigma, latent_mu, latent_logsigma
        return reconstruction_mu, reconstruction_logsigma, latent_mu, latent_logsigma


And the last, but not least! Place in the code where the most of the formulaes goes to - optimization objective. The objective for VAE has it's own name - variational lowerbound. And as for any lowerbound our intention is to maximize it. Here it is (for one sample $z$ per input $x$):

$$\mathcal{L} = -D_{KL}(q_{\phi}(z|x)||p(z)) + \log p_{\theta}(x|z) \to max$$

Your next task is to implement two functions that compute KL-divergence and the second term - log-likelihood of an output. Here is some necessary math for your convenience:

$$D_{KL} = -\frac{1}{2}\sum_{i=1}^{dimZ}(1+log(\sigma_i^2)-\mu_i^2-\sigma_i^2)$$
$$\log p_{\theta}(x|z) = \sum_{i=1}^{dimX}\log p_{\theta}(x_i|z)=\sum_{i=1}^{dimX} \log \Big( \frac{1}{\sigma_i\sqrt{2\pi}}e^{-\frac{(\mu_i-x)^2}{2\sigma_i^2}} \Big)=...$$

Don't forget in the code that you are using $\log\sigma$ as variable. Explain, why not $\sigma$?

I used log standard deviation because it can take any real value, while standard deviation must be positive: `sigma = exp(logsigma)`. Here `logsigma` means log standard deviation, not log variance; therefore `variance = exp(2 * logsigma)`. I summed KL over latent coordinates and log-likelihood over pixels, then averaged their difference over the batch. Minimizing `KL - log_likelihood` maximizes the ELBO. The Gaussian negative log-likelihood can be negative for continuous data; it is not an MSE or a classification probability.

In [ ]:
#changed
def KL_divergence(mu, logsigma):
    # summed over latent coordinates and kept one value per image
    return 0.5 * (mu.square() + (2 * logsigma).exp() - 1 - 2 * logsigma).sum(dim=1)


def log_likelihood(x, mu, logsigma):
    # used Gaussian pixel densities with log standard deviations
    log_prob = -0.5 * np.log(2 * np.pi) - logsigma - 0.5 * (x - mu).square() * (-2 * logsigma).exp()
    return log_prob.flatten(start_dim=1).sum(dim=1)


def loss_vae(x, mu_gen, logsigma_gen, mu_z, logsigma_z):
    return (KL_divergence(mu_z, logsigma_z) - log_likelihood(x, mu_gen, logsigma_gen)).mean()


# checked simple cases without training
zeros_z = torch.zeros(2, dimZ)
assert torch.allclose(KL_divergence(zeros_z, zeros_z), torch.zeros(2))
assert torch.allclose(KL_divergence(torch.ones_like(zeros_z), zeros_z), torch.full((2,), dimZ / 2))
zeros_x = torch.zeros(2, 1, 64, 64)
expected_log_prob = torch.full((2,), -0.5 * 64 * 64 * np.log(2 * np.pi))
assert torch.allclose(log_likelihood(zeros_x, zeros_x, zeros_x), expected_log_prob)
assert torch.allclose(loss_vae(zeros_x, zeros_x, zeros_x, zeros_z, zeros_z), -expected_log_prob.mean())
print('Loss checks passed.')


And train the model:

In [ ]:
#changed
# train your variational autoencoder
# visualize progress in reconstruction and loss decay
torch.manual_seed(1337)
vae = VAE().to(device)
# checked both Gaussian parameter pairs before training
with torch.no_grad():
    test_outputs = vae(torch.zeros(2, 1, 64, 64, device=device))
assert [tuple(value.shape) for value in test_outputs] == [
    (2, 1, 64, 64), (2, 1, 64, 64), (2, dimZ), (2, dimZ)
]
assert all(torch.isfinite(value).all().item() for value in test_outputs)
vae_optimizer = optim.Adam(vae.parameters())
num_epochs = 10
vae_history = {'train_loss': [], 'val_loss': [], 'val_kl': [], 'val_nll': []}
best_vae_loss = float('inf')

for epoch in range(num_epochs):
    start_time = time.time()
    vae.train()
    train_loss, train_count = 0.0, 0
    for images, _ in tqdm(train_loader, desc=f'VAE {epoch + 1}/{num_epochs}', leave=False):
        images = images.to(device)
        vae_optimizer.zero_grad()
        mu_gen, logsigma_gen, mu_z, logsigma_z = vae(images)
        loss = loss_vae(images, mu_gen, logsigma_gen, mu_z, logsigma_z)
        assert torch.isfinite(loss).item(), 'VAE loss became non-finite.'
        loss.backward()
        # clipped gradients to limit unstable updates
        nn.utils.clip_grad_norm_(vae.parameters(), max_norm=5.0, error_if_nonfinite=True)
        vae_optimizer.step()
        train_loss += loss.item() * images.size(0)
        train_count += images.size(0)

    vae.eval()
    val_kl, val_nll, val_count = 0.0, 0.0, 0
    with torch.no_grad():
        for images, _ in val_loader:
            images = images.to(device)
            mu_gen, logsigma_gen, mu_z, logsigma_z = vae(images)
            val_kl += KL_divergence(mu_z, logsigma_z).sum().item()
            val_nll -= log_likelihood(images, mu_gen, logsigma_gen).sum().item()
            val_count += images.size(0)
        # used the posterior mean for consistent reconstruction previews
        preview_mu, _ = vae.encode(preview_images)
        preview_reconstruction, _ = vae.decode(preview_mu)

    train_loss /= train_count
    val_kl /= val_count
    val_nll /= val_count
    val_loss = val_kl + val_nll
    assert np.isfinite(val_loss), 'VAE validation loss became non-finite.'
    vae_history['train_loss'].append(train_loss)
    vae_history['val_loss'].append(val_loss)
    vae_history['val_kl'].append(val_kl)
    vae_history['val_nll'].append(val_nll)
    if val_loss < best_vae_loss:
        best_vae_loss = val_loss
        torch.save(vae.state_dict(), checkpoint_dir / 'vae_best.pt')
    torch.save({
        'epoch': epoch + 1, 'model_state_dict': vae.state_dict(),
        'optimizer_state_dict': vae_optimizer.state_dict(), 'history': vae_history,
    }, checkpoint_dir / 'vae_last.pt')

    print(f'VAE epoch {epoch + 1}: train loss = {train_loss:.2f}, val loss = {val_loss:.2f}, '
          f'val KL = {val_kl:.2f}, val NLL = {val_nll:.2f}, time = {time.time() - start_time:.1f}s')
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(range(1, epoch + 2), vae_history['train_loss'], label='train')
    axes[0].plot(range(1, epoch + 2), vae_history['val_loss'], label='validation')
    axes[0].set(xlabel='epoch', ylabel='negative ELBO per image', title='VAE loss')
    axes[0].legend()
    grid = make_grid(torch.cat([preview_images, preview_reconstruction]).cpu(), nrow=8)
    axes[1].imshow(grid.permute(1, 2, 0))
    axes[1].set_title('VAE: originals (top), reconstructions (bottom)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

# restored the checkpoint with the lowest validation negative ELBO
vae.load_state_dict(torch.load(checkpoint_dir / 'vae_best.pt', map_location=device, weights_only=True))
vae.eval()
print(f'Best VAE validation negative ELBO: {best_vae_loss:.2f}')


In [ ]:
#changed
images, _ = list(islice(val_loader, 1))[0]
vae.eval()
with torch.no_grad():
    # decoded the posterior mean instead of adding random latent noise
    mu_z, _ = vae.encode(images.to(device))
    mu_gen, _ = vae.decode(mu_z)
reconstruction = mu_gen[:16].cpu().clamp(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(make_grid(images[:16], nrow=4).permute(1, 2, 0))
axes[0].set_title('Originals')
axes[1].imshow(make_grid(reconstruction, nrow=4).permute(1, 2, 0))
axes[1].set_title('VAE reconstructions from posterior means')
for ax in axes:
    ax.axis('off')
plt.show()
plt.close(fig)


And finally sample from VAE.

In [ ]:
#changed
# Sample some images from the learned distribution
# 1) Sample z ~ N(0,1)
# 2) Sample from N(decoder_mu(z), decoder_sigma(z))
vae.eval()
with torch.no_grad():
    z = torch.randn(25, dimZ, device=device)
    mu_gen, logsigma_gen = vae.decode(z)
    samples = vae.gaussian_sampler(mu_gen, logsigma_gen)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
# clipped only for display; kept the Gaussian samples unchanged
axes[0].imshow(make_grid(mu_gen.cpu().clamp(0, 1), nrow=5).permute(1, 2, 0))
axes[0].set_title('VAE decoder means')
axes[1].imshow(make_grid(samples.cpu().clamp(0, 1), nrow=5).permute(1, 2, 0))
axes[1].set_title('VAE Gaussian samples')
for ax in axes:
    ax.axis('off')
plt.show()
plt.close(fig)

# compared both best checkpoints using the same validation reconstruction metric
autoencoder.eval()
vae.eval()
ae_squared_error, vae_squared_error, pixel_count = 0.0, 0.0, 0
with torch.no_grad():
    for images, _ in val_loader:
        images = images.to(device)
        ae_reconstruction, _ = autoencoder(images)
        latent_mu, _ = vae.encode(images)
        vae_reconstruction, _ = vae.decode(latent_mu)
        ae_squared_error += (ae_reconstruction - images).square().sum().item()
        vae_squared_error += (vae_reconstruction - images).square().sum().item()
        pixel_count += images.numel()
validation_mse = {'AE': ae_squared_error / pixel_count, 'VAE': vae_squared_error / pixel_count}
for name, mse in validation_mse.items():
    print(f'{name}: validation reconstruction MSE per pixel = {mse:.6f}')
np.savez(
    checkpoint_dir / 'training_metrics.npz',
    ae_train_mse=ae_history['train_mse'], ae_val_mse=ae_history['val_mse'],
    vae_train_loss=vae_history['train_loss'], vae_val_loss=vae_history['val_loss'],
    vae_val_kl=vae_history['val_kl'], vae_val_nll=vae_history['val_nll'],
    final_ae_mse=validation_mse['AE'], final_vae_mse=validation_mse['VAE'],
)


Even if in practice you do not see the much difference between AE and VAE, or VAE is even worse, the little bayesian inside you should be jumping for joy right now.

In VAE you can truly sample from image distribution $p(x)$, while in AE there is no easy and correct way to do it.

### comparison notes

The final cell reports validation MSE per pixel for both selected models, using the VAE posterior mean for deterministic reconstruction. AE MSE and VAE negative ELBO have different scales and meanings, so their training losses should not be compared directly. Lower reconstruction MSE alone also does not establish better random generation.

After the GPU run, compare the saved curves and images: which model reconstructed face details better, whether random latent vectors produced recognizable faces, and whether pixel noise changed the VAE samples noticeably. No measured scores or visual conclusions are claimed here before execution. The optional smile-arithmetic bonus below has not been implemented.

# If you made it here - Congrats! (and here's an optional bonus task)

If you managed to tune your autoencoders to converge and learn something about the world, now it's time to make fun out of it. As you may have noticed, there are face attributes in dataset. We're interesting in "Smiling" column (index 31 of the attribute vector), but feel free to try others as well! Here is the first task:

1) Extract the "Smilling" attribute and create a two sets of images: 10 smiling faces and 10 non-smiling ones.

2) Compute latent representations for each image in "smiling" set and average those latent vectors. Do the same for "non-smiling" set. You have found **"vector representation"** of the "smile" and "no smile" attribute.

3) Compute the difference: "smile" vector minus "non-smile" vector.

3) Now check if **"feature arithmetics"** works. Sample a face without smile, encode it and add the diff from p. 3. Check if it works with both AE and VAE.